In [ ]:
import numpy as np

def modularity_optimization(A, rng, gamma=1.0):
    """
    Optimize the modularity of a network by moving nodes between communities.

    A       : symmetric weighted adjacency matrix (n × n), zero diagonal
    rng     : numpy random number generator (for reproducibility)
    gamma   : resolution; higher -> more, smaller communities
    Returns : membership (community label for each node)
    """
    n = len(A)
    k = A.sum(axis=1)
    two_m = A.sum()
    c = np.arange(n)              # every node in its own community
    sigma_tot = k.copy()          # so each community's total is just that node's degree

    improved = True
    while improved:               # repeat sweeps until no node wants to move
        improved = False
        for i in rng.permutation(n):
            if move_node(i, A, k, c, sigma_tot, two_m, gamma):
                improved = True
    return c

def community_aggregation(A, c):
    """Aggregate communities into super-nodes.
    Returns the new adjacency matrix and the renumbered labels (node -> super-node)."""
    c = np.unique(c, return_inverse=True)[1]     # relabel to 0..nC-1
    nC = c.max() + 1
    A_new = np.zeros((nC, nC))
    for i in range(len(A)):
        for j in range(len(A)):
            A_new[c[i], c[j]] += A[i, j]
    return A_new, c

def move_node(i, A, k, c, sigma_tot, two_m, gamma=1.0):
    """Move node i to the neighbouring community with the largest modularity gain.
    Updates c and sigma_tot in place. Returns True if i changed community."""
    old = c[i]

    # 1. Take i out of its community
    sigma_tot[old] -= k[i]

    # 2. Weight from i to each neighbouring community: k_i,in(C)
    nb = np.nonzero(A[i])[0]
    nb = nb[nb != i]                                   # ignore self-loop
    k_i_in = np.bincount(c[nb], weights=A[i, nb], minlength=len(sigma_tot))

    # 3. Gain for each candidate community; default is going back to the old one
    best, best_gain = old, 0.0
    for C in np.unique(c[nb]):
        gain = k_i_in[C] - gamma * sigma_tot[C] * k[i] / two_m
        if gain > best_gain + 1e-12:                   # tolerance against float noise
            best, best_gain = C, gain

    # 4. Put i in the best community
    c[i] = best
    sigma_tot[best] += k[i]
    return best != old


def compute_network_modularity(A: np.ndarray, c: np.ndarray) -> float:
    two_m = A.sum()
    k = A.sum(axis=1)
    n = len(A)
    return sum((A[i, j] - k[i] * k[j] / two_m) * (c[i] == c[j])
               for i in range(n) for j in range(n)) / two_m


def louvain_algorithm(A, gamma=1.0, seed=None, verbose=True):
    """Louvain community detection from scratch.

    A       : symmetric weighted adjacency matrix (n × n), zero diagonal
    gamma   : resolution; higher -> more, smaller communities
    seed    : random seed for the node visiting order (for reproducibility)
    Returns : membership (community label for each original node), final modularity Q
    """
    rng = np.random.default_rng(seed)
    A = np.array(A, dtype=float)
    A_orig = A.copy()                        # Q is always measured on the original graph
    membership = np.arange(len(A))           # original node -> current super-node
    level = 0

    if verbose:
        print(f"level 0: {len(A)} nodes, Q = {compute_network_modularity(A_orig, membership):.4f}")

    while True:
        # Phase 1: move nodes (or super-nodes) between communities
        c = modularity_optimization(A, rng, gamma)

        # Stop when phase 1 merged nothing: every node is still on its own
        if len(np.unique(c)) == len(A):
            break

        # Phase 2: turn communities into super-nodes
        A, c = community_aggregation(A, c)
        membership = c[membership]
        level += 1

        if verbose:
            print(f"level {level}: {len(A)} communities, "
                  f"Q = {compute_network_modularity(A_orig, membership):.4f}")

    return membership, compute_network_modularity(A_orig, membership)

    

   


In [12]:
import networkx as nx

G = nx.karate_club_graph()
A = nx.to_numpy_array(G)          # 34 × 34 weighted adjacency matrix (weight=None for unweighted)
n = len(A)

print(f"nodes: {n}, edges: {int((A > 0).sum() / 2)}, total weight m: {A.sum() / 2:.0f}")

# Before: every node in its own community
c0 = np.arange(n)
print(c0)
print(f"before: {len(np.unique(c0))} communities, Q = {compute_network_modularity(A, c0):.4f}")

# After phase 1
rng = np.random.default_rng(7)
c1 = modularity_optimization(A, rng)
c1 = np.unique(c1, return_inverse=True)[1]          # renumber labels to 0, 1, 2, ...
print(f"after:  {len(np.unique(c1))} communities, Q = {compute_network_modularity(A, c1):.4f}, "
      f"sizes = {np.bincount(c1).tolist()}")

# Checks against networkx
comms = [set(np.where(c1 == C)[0]) for C in np.unique(c1)]
print(f"networkx Q of our partition: {nx.community.modularity(G, comms):.4f}")   # must match ours
nxl = nx.community.louvain_communities(G, seed=6)
print(f"networkx full Louvain: {len(nxl)} communities, Q = {nx.community.modularity(G, nxl):.4f}")

# The real-world split
club = np.array([G.nodes[v]["club"] == "Officer" for v in G]).astype(int)
print(f"real split: Q = {compute_network_modularity(A, club):.4f}")

nodes: 34, edges: 78, total weight m: 231
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33]
before: 34 communities, Q = -0.0511
after:  7 communities, Q = 0.3663, sizes = [11, 2, 2, 3, 6, 4, 6]
networkx Q of our partition: 0.3663
networkx full Louvain: 4 communities, Q = 0.4439
real split: Q = 0.3914
